# EKG Assistant — Phase 3 Training on Kaggle GPU
Settings: **Accelerator = GPU (T4 or P100)**, **Internet = On**.
Inputs: the `ekg_code.zip` dataset and a PTB-XL dataset (100Hz records + CSVs).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q wfdb onnx onnxruntime

In [ ]:
from pathlib import Path
import shutil

inputs = Path('/kaggle/input')
csv = next(inputs.rglob('ptbxl_database.csv'))
PTBXL_DIR = csv.parent
assert (PTBXL_DIR / 'records100').is_dir(), f'records100 not found next to {csv}'
code_src = next(inputs.rglob('train_improved.py')).parent
CODE_DIR = Path('/kaggle/working/code')
shutil.copytree(code_src, CODE_DIR, dirs_exist_ok=True)
print('PTB-XL:', PTBXL_DIR)
print('Code  :', CODE_DIR, sorted(p.name for p in CODE_DIR.iterdir()))

## Train
Progress (%), ETA and validation AUPRC print once per epoch. Early stopping after 5 epochs without improvement.

In [ ]:
!cd /kaggle/working/code && EKG_PTBXL_DIR={PTBXL_DIR} PYTHONPATH=. python -u train_improved.py \
    --epochs 50 --batch-size 64 --patience 5 --output-dir /kaggle/working/output

## Package results
Download `phase3_output.zip` from the **Output** panel and extract it into `models/candidates/` locally.

In [ ]:
shutil.make_archive('/kaggle/working/phase3_output', 'zip', '/kaggle/working/output')
for p in sorted(Path('/kaggle/working/output').iterdir()):
    print(f'{p.name:35s} {p.stat().st_size / 1e6:8.2f} MB')